# Model Serving

Notebook for **Chapter 7** of the *mlops* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/mlops/ch07.html

Tip: change a number, predict the output, then run the cell again.

## 7.1 A Prediction Service with Validation and Health Checks

In [ ]:
N_FEAT = 8
class ValidationError(Exception): pass

def validate(body):
    if not isinstance(body, dict) or "features" not in body:
        raise ValidationError("field 'features' is required")
    f = body["features"]
    if not isinstance(f, list) or len(f) != N_FEAT:
        raise ValidationError(f"features must be a list of {N_FEAT} numbers")
    if not all(isinstance(x, (int, float)) and not isinstance(x, bool) for x in f):
        raise ValidationError("features must be numbers")
    if not all(abs(x) < 1e6 for x in f):
        raise ValidationError("feature out of range")
    return f

In [ ]:
# not run here
from fastapi import FastAPI
from pydantic import BaseModel, Field, conlist
import joblib, numpy as np

app = FastAPI(title="churn-api")
model = joblib.load("model_v1.joblib")

class Request(BaseModel):
    features: conlist(float, min_length=8, max_length=8)

class Response(BaseModel):
    probability: float = Field(ge=0, le=1)
    model_version: str

@app.get("/health")
def health(): return {"status": "ok"}

@app.post("/v1/predict", response_model=Response)
def predict(req: Request):
    p = float(model.predict_proba(np.array([req.features]))[0, 1])
    return Response(probability=round(p, 4), model_version="v1")

## 7.2 Batching: Why One Row at a Time Is Wasteful

## 7.3 Model Formats and the Pickle Risk

In [ ]:
import pickle, json
class Evil:
    def __reduce__(self): return (print, ("!!! code ran just by loading this pickle !!!",))
blob = pickle.dumps(Evil())
pickle.loads(blob)
print("JSON loads only data:", json.loads('{"w":[0.1,0.2]}'))

In [ ]:
# not run here
from skl2onnx import to_onnx
import onnxruntime as ort, numpy as np
onx = to_onnx(model, X[:1].astype(np.float32))      # convert scikit-learn model
open("churn.onnx", "wb").write(onx.SerializeToString())
sess = ort.InferenceSession("churn.onnx")
out = sess.run(None, {sess.get_inputs()[0].name: X[:5].astype(np.float32)})

## 7.4 Serving Frameworks and LLM Serving (Concepts)

## 7.5 Latency Budgets, Queues and Autoscaling

So far you sized services by throughput alone. Real users feel latency, and latency depends on how full the service is. This section gives you three tools you will use in design reviews: a latency budget, Little's law, and a queue model that predicts the wait time at a given load.

In [ ]:
# Little's law: L = lambda * W  (average number in the system)
lam = 250          # requests per second
W = 0.080          # seconds each request spends in the service
L = lam * W
print("in flight:", L)
# if each request holds 1 worker slot, you need at least L slots, plus headroom
import math
print("slots at 60% target:", math.ceil(L / 0.6))

in flight: 20.0
slots at 60% target: 34


In [ ]:
import math
# M/M/1 queue: one worker, random arrivals (rate lam), random service (rate mu)
mu = 100.0                       # requests per second one replica can finish
print("util  mean_wait_ms  p95_ms   p99_ms")
for u in (0.5, 0.7, 0.8, 0.9, 0.95, 0.99):
    lam = u * mu
    rate = mu - lam              # time in system ~ Exponential(rate)
    mean = 1000 / rate
    p95 = 1000 * math.log(1/0.05) / rate
    p99 = 1000 * math.log(1/0.01) / rate
    print(f"{u:.2f}  {mean:10.1f}  {p95:7.1f}  {p99:7.1f}")

util  mean_wait_ms  p95_ms   p99_ms
0.50        20.0     59.9     92.1
0.70        33.3     99.9    153.5
0.80        50.0    149.8    230.3
0.90       100.0    299.6    460.5
0.95       200.0    599.1    921.0
0.99      1000.0   2995.7   4605.2


In [ ]:
import math
def erlang_c(c, a):
    """Chance an arriving request must wait. c servers, offered load a = lam/mu."""
    top = (a**c / math.factorial(c)) * (c / (c - a))
    bottom = sum(a**k / math.factorial(k) for k in range(c)) + top
    return top / bottom

def p99_ms(c, lam, mu):
    """Rough p99 of time in system for M/M/c (wait tail plus mean service)."""
    a = lam / mu
    pw = erlang_c(c, a)
    # waiting time tail: P(W > t) = pw * exp(-(c*mu - lam) t)
    t99_wait = max(0.0, math.log(pw / 0.01) / (c * mu - lam))
    return 1000 * (t99_wait + 1 / mu)

lam, mu = 400.0, 100.0          # 400 req/s peak, one replica does 100 req/s
for c in range(5, 10):
    util = lam / (c * mu)
    print(c, f"util={util:.2f}", f"P(wait)={erlang_c(c, lam/mu):.3f}", f"p99~{p99_ms(c, lam, mu):.0f} ms")

5 util=0.80 P(wait)=0.554 p99~50 ms
6 util=0.67 P(wait)=0.285 p99~27 ms
7 util=0.57 P(wait)=0.135 p99~19 ms
8 util=0.50 P(wait)=0.059 p99~14 ms
9 util=0.44 P(wait)=0.024 p99~12 ms


In [ ]:
import math
# Kubernetes HPA rule: desired = ceil(current * currentMetric / targetMetric)
# The controller ignores changes while the ratio is within 10% of 1 (default tolerance).
def hpa(current, metric, target, tolerance=0.10):
    ratio = metric / target
    if abs(ratio - 1) <= tolerance:
        return current
    return math.ceil(current * ratio)
print(hpa(4, 90, 60))     # CPU at 90%, target 60%
print(hpa(4, 30, 60))     # CPU at 30%
print(hpa(10, 65, 60))    # ratio 1.083, inside the tolerance band
print(hpa(10, 75, 60))    # ratio 1.25

6
2
10
13


## 7.6 LLM Serving Arithmetic: KV Cache, Bandwidth and Batching

Serving an LLM is different from serving a churn model because the model writes its answer one token at a time, and it must remember everything it has read. The memory for that remembering, the KV cache, is usually what limits how many users one GPU can serve. This section gives you the arithmetic. Every number below is computed by code you can run, but I did not run a real GPU server here: the figures are calculations, not benchmarks. Treat them as upper bounds and measure on your hardware (not run here).

In [ ]:
# KV cache bytes per token = 2 (K and V) x layers x kv_heads x head_dim x bytes per number
def kv_per_token(layers, kv_heads, head_dim, nbytes=2):
    return 2 * layers * kv_heads * head_dim * nbytes

models = {
    "7B, full multi-head (32 layers, 32 KV heads)": (32, 32, 128),
    "8B, grouped-query (32 layers, 8 KV heads)": (32, 8, 128),
    "70B, grouped-query (80 layers, 8 KV heads)": (80, 8, 128),
}
for name, (L, h, d) in models.items():
    b = kv_per_token(L, h, d)
    print(f"{name}: {b} bytes = {b/1024:.0f} KiB per token, 4096 tokens = {b*4096/2**30:.2f} GiB")

7B, full multi-head (32 layers, 32 KV heads): 524288 bytes = 512 KiB per token, 4096 tokens = 2.00 GiB
8B, grouped-query (32 layers, 8 KV heads): 131072 bytes = 128 KiB per token, 4096 tokens = 0.50 GiB
70B, grouped-query (80 layers, 8 KV heads): 327680 bytes = 320 KiB per token, 4096 tokens = 1.25 GiB


In [ ]:
# How many users fit? 80 GB GPU, keep 10% free, fp16 weights
gpu = 80e9 * 0.90
weights = 8.03e9 * 2                  # 8.03 billion parameters x 2 bytes
per_tok = 2 * 32 * 8 * 128 * 2        # 131072 bytes
budget = gpu - weights
print("KV budget GB:", round(budget / 1e9, 1))
for ctx in (2048, 8192, 32768):
    per_user = per_tok * ctx
    print(ctx, "tokens: per user", round(per_user / 1e9, 2), "GB, users:", int(budget // per_user))
# same model with INT8 weights (1 byte) and fp8 KV cache (1 byte)
budget8 = gpu - 8.03e9 * 1
for ctx in (2048, 8192, 32768):
    per_user = per_tok // 2 * ctx
    print("int8 weights + fp8 KV,", ctx, "tokens: users:", int(budget8 // per_user))

KV budget GB: 55.9
2048 tokens: per user 0.27 GB, users: 208
8192 tokens: per user 1.07 GB, users: 52
32768 tokens: per user 4.29 GB, users: 13
int8 weights + fp8 KV, 2048 tokens: users: 476
int8 weights + fp8 KV, 8192 tokens: users: 119
int8 weights + fp8 KV, 32768 tokens: users: 29


In [ ]:
# Decode speed is limited by memory bandwidth: each new token reads all weights once.
bandwidth = 3.35e12                   # bytes/s, an H100-class GPU (3.35 TB/s)
for name, nbytes in (("8B fp16", 16.06e9), ("8B int8", 8.03e9), ("8B 4-bit", 4.0e9)):
    one_stream = bandwidth / nbytes
    print(f"{name}: upper bound {one_stream:.0f} tokens/s for ONE stream")
# batching: 32 streams share one read of the weights (ignoring KV reads)
print("32 streams, fp16:", round(32 * bandwidth / 16.06e9), "tokens/s total (upper bound)")

8B fp16: upper bound 209 tokens/s for ONE stream
8B int8: upper bound 417 tokens/s for ONE stream
8B 4-bit: upper bound 838 tokens/s for ONE stream
32 streams, fp16: 6675 tokens/s total (upper bound)


In [ ]:
# PagedAttention vs reserving the maximum length for every request
max_len, per_tok = 4096, 131072
actual = [300, 900, 150, 2500, 600, 1200, 400, 700]     # real tokens used by 8 requests
reserved = len(actual) * max_len * per_tok
paged = sum(-(-t // 16) * 16 for t in actual) * per_tok  # blocks of 16 tokens
print("reserved GB:", round(reserved / 1e9, 2))
print("paged GB:", round(paged / 1e9, 2))
print("saving:", f"{1 - paged/reserved:.0%}")

reserved GB: 4.29
paged GB: 0.89
saving: 79%


In [ ]:
# Speculative decoding: a small draft model proposes k tokens, the big model checks them in one pass.
# If each draft token is accepted with probability a, expected tokens per big-model pass:
def expected(a, k):
    return (1 - a ** (k + 1)) / (1 - a)
for a in (0.5, 0.7, 0.9):
    print(a, [round(expected(a, k), 2) for k in (2, 4, 8)])

0.5 [1.75, 1.94, 2.0]
0.7 [2.19, 2.77, 3.2]
0.9 [2.71, 4.1, 6.13]


## Chapter recap

## Practice set

**1.** (Easy) A service answers 4,850 of 5,000 requests with 200 and the rest with 422. What is the error rate?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/mlops/ch07.html